In [1]:
import ollama 
import os
from tqdm import tqdm
import json
import signal
import argparse
import wandb
import pandas as pd
import copy

import sys
from collections import defaultdict
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score,confusion_matrix
from pathlib import Path

In [2]:
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [3]:
sys.argv = [
    'notebook',  
    '--modelname', 'llama3.2-vision:90b', #'llama3.2-vision:90b', #'qwen2:72b',  
    '--data', '/root/home/data',
    '--data_path_files','/mnt/jacket/WACV-2025-Workshop-ViGIR',
    '--results_dir', '/mnt/jacket/WACV-2025-Workshop-ViGIR/results/baseline',
    '--timeout', '20',
    '--model_unloading'
]

In [4]:
parser = argparse.ArgumentParser(description="A script to run V-LLMs on different image classification datasets")

In [5]:
parser.add_argument("--modelname", type=str, required=True, help="The name of the V-LLM model")
parser.add_argument("--data", type=str, required=True, help="Path to the data")
parser.add_argument("--data_path_files", type=str, required=True, help="Path to the image data dir")
parser.add_argument("--results_dir", type=str, required=True, help="Folder name to save results")
parser.add_argument("--timeout", type=int, default=40, help="time out duration to skip one sample")
parser.add_argument("--model_unloading", action="store_true", help="Enables unloading mode. Every 100 sampels it unloades the model from the GPU to avoid carshing.")

args = parser.parse_args()

In [6]:
valid_file=os.path.join(args.data_path_files,'Validation_Set.json')
test_file=os.path.join(args.data_path_files,'Testing_Set.json')

with open(valid_file, 'r') as file:
    valid = json.load(file) 

with open(test_file, 'r') as file:
    test = json.load(file) 

In [7]:
def display_image(image_path):
    img = mpimg.imread(image_path)
    plt.figure(figsize=(20, 20))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [8]:
directory_path=Path(args.data)

file_dict = {}


for file in directory_path.iterdir():
    if file.is_file() and 'collage' in file.name:
        parts = file.stem.split('_')  
        number = parts[-1]
        if number.isdigit():
            file_dict[int(number)] = str(file)

In [9]:
# # Set up the run 
# run = wandb.init(
#     entity="ramytrm",
#     project=f"wacv-2025-{args.data}",
#     name="run_test_" + args.data+"-"+args.modelname+"-"+args.subset
# )

In [10]:
old_questions = [
    "Across the images, do you see a low point in the terrain where a temporary gully or depression begins to form without a permanent water feature? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Do narrow, winding paths or channels appear and become more defined across the sequence? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Over time, do any linear depressions or ruts become more pronounced along natural drainage lines or slopes? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Does a narrow, shallow channel become visibly deeper or more indented into the soil over the image sequence? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Are there areas where soil appears disturbed or vegetation is removed, showing progression over the images? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Across the sequence, does a specific path lack vegetation, suggesting an evolving or emerging channel? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Does the texture of the soil appear to change, becoming coarser or showing small pebbles in areas that develop into channels? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Can you see any clear starting and ending points for these channels becoming more evident over time? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Do small rills or grooves indicating water flow appear or deepen across the images? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Is there a gradual exposure of lighter-colored soil as the gully forms? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Are there sediment accumulations forming at the lower ends of channels as time progresses? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Across the images, do signs of water activity, like soil clumps or crusting, appear or intensify in specific areas? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Do shallow side walls along the channel areas become more defined as the sequence progresses? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Do deposits of silt or small debris appear and accumulate along any emerging channel paths? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Can you observe branching patterns that resemble temporary streams forming or intensifying over time? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Do areas with compacted soil or minor slope collapse become more noticeable along the developing channel edges? Please answer with YES or NO Only. DO NOT mention the reason.",
    "Are there indications of nearby human activity, such as tillage or machinery tracks, influencing the formation or expansion of these channels? Please answer with YES or NO Only. DO NOT mention the reason."
]


In [11]:
baseline_question = ["Given this collage of six images of the exact same area, collected over a period of 10 years. Are there any ephemeral gully appearances? Answer by Yes or No and nothing else."]

In [12]:

prompts_all = [
    "Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, are there winding paths that become intermittent recurrent? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, are there any linear depressions or ruts which appear more pronounced along natural drainage lines or slopes? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, are there narrow and shallow channels which appear intermittently deeper or more indented into the soil? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, are there areas where soil appears disturbed or vegetation is removed? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, does a specific path lack vegetation, suggesting an evolving or emerging channel? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, are there varying types and levels of coarseness in the texture of the soil? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, are there clear starting and ending points of potential channels? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, are there small rills or grooves indicating water flow? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, is there a varying exposure of lighter or darker colored soil? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, are there sediment accumulations forming? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, are there signs of water activity, like soil clumps or crusting, that appear or intensify in specific areas? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, are there any branching patterns that resemble temporary streams? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, are there indications of nearby human activity, such as tillage or machinery tracks? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, do you see any sign of water flow patterns across the field in multiple images? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, do you see any edges in the images indicating removal of soil along the water pathway? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, do you see any cuts in the soil associated with water flow across the field? Answer with yes or no only!",
    "Given these six images of the exact same area and collected over a period of 10 years, do you see any indication of human activity such as tillage that is not naturally happening in the field? Answer with yes or no only!"
]

prompts_all




['Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!',
 'Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!',
 'Given these six images of the exact same area and collected over a period of 10 years, are there winding paths that become intermittent recurrent? Answer with yes or no only!',
 'Given these six images of the exact same area and collected over a period of 10 years, are there any linear depressions or ruts which appear more pronounced along natural drainage lines or slopes? Answer with yes or no only!',
 'Given these six images of the exact same area and collected over a period of 10 years, are there narrow and shallow channels which appear intermittently deeper or more indented into the soil? Answer with yes or no only!',
 'Given these six images of the exact same are

In [13]:
# choose the experiment
num_questions = 3
if num_questions == 3:
    prompts = []
    prompts.append(prompts_all[1])
    prompts.append(prompts_all[2])
    prompts.append(prompts_all[13])
prompts

prompts = prompts_all

In [14]:
len(prompts)

19

In [15]:
model_name = args.modelname #'llama3.2-vision:90b'
ollama.pull(model_name)

timeout_duration = args.timeout

options= {  # new
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048, # must be set, otherwise slightly random output
        }

model_labels = {}
count = 0

In [16]:
## Run for the valid set
count=0
saving_response={}

for key,info in tqdm(valid.items()):
    path=file_dict[int(key)]
    print(key,path)
    count+=1
    saving_response[key]=[]
    for question in prompts:
        response = ollama.generate(model=model_name, prompt=question, images=[path], options=options)
        saving_response[key].append([info,question,response['response']])
        print(key,path,info,question)
        print(response['response'])
        

#with open(os.path.join(args.results_dir,f'valid_{model_name}.json'), "w") as file:
#    json.dump(saving_response, file)
    

  0%|                                                                                                                                                    | 0/310 [00:00<?, ?it/s]

356 /root/home/data/pos_collage_356.jpg
356 /root/home/data/pos_collage_356.jpg {'label': '4', 'labelers': ['Pankaj', 'Parth']} Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!
No.
356 /root/home/data/pos_collage_356.jpg {'label': '4', 'labelers': ['Pankaj', 'Parth']} Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!
No.
356 /root/home/data/pos_collage_356.jpg {'label': '4', 'labelers': ['Pankaj', 'Parth']} Given these six images of the exact same area and collected over a period of 10 years, are there winding paths that become intermittent recurrent? Answer with yes or no only!
No.
356 /root/home/data/pos_collage_356.jpg {'label': '4', 'labelers': ['Pankaj', 'Parth']} Given these six images of the exact same area and collected over a period of 10 years, are there any linear

  0%|▍                                                                                                                                       | 1/310 [05:25<27:57:10, 325.66s/it]

356 /root/home/data/pos_collage_356.jpg {'label': '4', 'labelers': ['Pankaj', 'Parth']} Given these six images of the exact same area and collected over a period of 10 years, do you see any indication of human activity such as tillage that is not naturally happening in the field? Answer with yes or no only!
No.
700 /root/home/data/pos_collage_700.jpg
700 /root/home/data/pos_collage_700.jpg {'label': '4', 'labelers': ['Dr.Lory', 'Yizheng']} Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!
No.
700 /root/home/data/pos_collage_700.jpg {'label': '4', 'labelers': ['Dr.Lory', 'Yizheng']} Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!
No.
700 /root/home/data/pos_collage_700.jpg {'label': '4', 'labelers': ['Dr.Lory', 'Yizheng']} Given these six images of the exact same area and co

  1%|▉                                                                                                                                       | 2/310 [05:35<11:57:57, 139.86s/it]

...

73 /root/home/data/neg_collage_73.jpg {'label': '4', 'labelers': ['Parth', 'Yizheng']} Given these six images of the exact same area and collected over a period of 10 years, do you see any indication of human activity such as tillage that is not naturally happening in the field? Answer with yes or no only!
No.
261 /root/home/data/neg_collage_261.jpg
261 /root/home/data/neg_collage_261.jpg {'label': '0', 'labelers': ['Brian', 'Yizheng']} Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!
No.
261 /root/home/data/neg_collage_261.jpg {'label': '0', 'labelers': ['Brian', 'Yizheng']} Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!
No.
261 /root/home/data/neg_collage_261.jpg {'label': '0', 'labelers': ['Brian', 'Yizheng']} Given these six images of the exact same area and collected

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌| 309/310 [51:19<00:08,  8.94s/it]

261 /root/home/data/neg_collage_261.jpg {'label': '0', 'labelers': ['Brian', 'Yizheng']} Given these six images of the exact same area and collected over a period of 10 years, do you see any indication of human activity such as tillage that is not naturally happening in the field? Answer with yes or no only!
No.
292 /root/home/data/neg_collage_292.jpg
292 /root/home/data/neg_collage_292.jpg {'label': '0', 'labelers': ['Ali', 'Yizheng']} Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!
No.
292 /root/home/data/neg_collage_292.jpg {'label': '0', 'labelers': ['Ali', 'Yizheng']} Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!
No.
292 /root/home/data/neg_collage_292.jpg {'label': '0', 'labelers': ['Ali', 'Yizheng']} Given these six images of the exact same area and collected ove

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 310/310 [51:28<00:00,  9.96s/it]

292 /root/home/data/neg_collage_292.jpg {'label': '0', 'labelers': ['Ali', 'Yizheng']} Given these six images of the exact same area and collected over a period of 10 years, do you see any indication of human activity such as tillage that is not naturally happening in the field? Answer with yes or no only!
No.


In [17]:
print(os.path.join(args.results_dir,f'valid_19Q_{model_name}.json'))

/mnt/jacket/WACV-2025-Workshop-ViGIR/results/baseline/valid_19Q_llama3.2-vision:90b.json


In [18]:
with open(os.path.join(args.results_dir,f'valid_19Q_{model_name}.json'), "w") as file:
    json.dump(saving_response, file)

In [65]:
# Run for the testing set
count=0
saving_response={}

for key,info in tqdm(test.items()):
    path=file_dict[int(key)]
    print(key,path)
    count+=1
    saving_response[key]=[]
    for question in prompts:
        response = ollama.generate(model=model_name, prompt=question, images=[path], options=options)
        saving_response[key].append([info,question,response['response']])
        print(key,path,info,question)
        print(response['response'])
        

#with open(os.path.join(args.results_dir,f'test_{num_questions}Q_{model_name}.json'), "w") as file:
#    json.dump(saving_response, file)
    

  0%|                                                                                                              | 0/311 [00:00<?, ?it/s]

415 /root/home/data/pos_collage_415.jpg
415 /root/home/data/pos_collage_415.jpg {'label': '4', 'labelers': ['Ali', 'Dr.Lory']} Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!
No.
415 /root/home/data/pos_collage_415.jpg {'label': '4', 'labelers': ['Ali', 'Dr.Lory']} Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!
No.
415 /root/home/data/pos_collage_415.jpg {'label': '4', 'labelers': ['Ali', 'Dr.Lory']} Given these six images of the exact same area and collected over a period of 10 years, are there winding paths that become intermittent recurrent? Answer with yes or no only!
No.
415 /root/home/data/pos_collage_415.jpg {'label': '4', 'labelers': ['Ali', 'Dr.Lory']} Given these six images of the exact same area and collected over a period of 10 years, are there any linear dep

  0%|▎                                                                                                     | 1/311 [00:09<46:46,  9.05s/it]

415 /root/home/data/pos_collage_415.jpg {'label': '4', 'labelers': ['Ali', 'Dr.Lory']} Given these six images of the exact same area and collected over a period of 10 years, do you see any indication of human activity such as tillage that is not naturally happening in the field? Answer with yes or no only!
No.
1020 /root/home/data/neg_collage_1020.jpg
1020 /root/home/data/neg_collage_1020.jpg {'label': '0', 'labelers': ['Krystal', 'Parth']} Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!
No.
1020 /root/home/data/neg_collage_1020.jpg {'label': '0', 'labelers': ['Krystal', 'Parth']} Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!
No.
1020 /root/home/data/neg_collage_1020.jpg {'label': '0', 'labelers': ['Krystal', 'Parth']} Given these six images of the exact same area and c

  1%|▋                                                                                                     | 2/311 [00:18<46:24,  9.01s/it]

...

212 /root/home/data/pos_collage_212.jpg {'label': '4', 'labelers': ['David', 'Yizheng']} Given these six images of the exact same area and collected over a period of 10 years, do you see any indication of human activity such as tillage that is not naturally happening in the field? Answer with yes or no only!
No.
131 /root/home/data/neg_collage_131.jpg
131 /root/home/data/neg_collage_131.jpg {'label': '0', 'labelers': ['David', 'Parth']} Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!
No.
131 /root/home/data/neg_collage_131.jpg {'label': '0', 'labelers': ['David', 'Parth']} Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!
No.
131 /root/home/data/neg_collage_131.jpg {'label': '0', 'labelers': ['David', 'Parth']} Given these six images of the exact same area and collected ove

100%|███████████████████████████████████████████████████████████████████████████████████████████████████▋| 310/311 [46:23<00:08,  8.96s/it]

131 /root/home/data/neg_collage_131.jpg {'label': '0', 'labelers': ['David', 'Parth']} Given these six images of the exact same area and collected over a period of 10 years, do you see any indication of human activity such as tillage that is not naturally happening in the field? Answer with yes or no only!
No.
1133 /root/home/data/neg_collage_1133.jpg
1133 /root/home/data/neg_collage_1133.jpg {'label': '0', 'labelers': ['Ali', 'Yizheng']} Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!
No.
1133 /root/home/data/neg_collage_1133.jpg {'label': '0', 'labelers': ['Ali', 'Yizheng']} Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!
No.
1133 /root/home/data/neg_collage_1133.jpg {'label': '0', 'labelers': ['Ali', 'Yizheng']} Given these six images of the exact same area and collect

100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 311/311 [46:32<00:00,  8.98s/it]

1133 /root/home/data/neg_collage_1133.jpg {'label': '0', 'labelers': ['Ali', 'Yizheng']} Given these six images of the exact same area and collected over a period of 10 years, do you see any indication of human activity such as tillage that is not naturally happening in the field? Answer with yes or no only!
No.


In [66]:
print(os.path.join(args.results_dir,f'test_19Q_{model_name}.json'))

/mnt/jacket/WACV-2025-Workshop-ViGIR/results/baseline/test_19Q_llama3.2-vision:90b.json


In [67]:
with open(os.path.join(args.results_dir,f'test_19Q_{model_name}.json'), "w") as file:
    json.dump(saving_response, file)

In [20]:
ground_gully=0
ground_no_gully=0

for key,info in saving_response.items():
    
    for i in range(len(info)):
        if info[i][0]['label']==str(4):
            if info[i][2]=='Yes.':
                ground_gully+=1
        else:
            if info[i][2]=='Yes.':
                ground_no_gully+=1
 

In [21]:
print(ground_gully)
print(ground_no_gully)

0
0


In [ ]:
with open(os.path.join(args.results_dir,f'valid_llama3_90b.json'), 'r') as file:
    saving_response_valid = json.load(file)

In [26]:
with open(os.path.join(args.results_dir,f'test_{args.modelname}.json'), 'r') as file:
    saving_response_test = json.load(file)

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/jacket/WACV-2025-Workshop-ViGIR/results/baseline/test_llava-llama3:8b.json'

In [ ]:
sum_correct=0
output_ground_truth=[]
output_model=[]


for key,info in saving_response_test.items():
    sum_yes=0
    for i in range(len(info)):
        if info[i][2]=='Yes.':
            sum_yes+=1
            
    if info[i][0]['label']==str(4) and sum_yes>=2:
        sum_correct+=1
        
    elif info[i][0]['label']==str(0) and sum_yes<2:
        sum_correct+=1
        
sum_correct/len(saving_response_test)






In [31]:
saving_response_test = copy.deepcopy(saving_response)

In [32]:
output_ground_truth=[]
output_model=[]


for key,info in saving_response_test.items():
    sum_yes=0
    for i in range(len(info)):
        if info[i][2]=='Yes.':
            sum_yes+=1
    
    if info[i][0]['label']==str(4):
        output_ground_truth.append(1)
    else:
        output_ground_truth.append(0)
    
    
    if sum_yes>=2:
        output_model.append(1)
    else:
        output_model.append(0)
        
        


In [33]:
output_ground_truth

[1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,


In [ ]:
macro_f1 = f1_score(output_ground_truth, output_model, average='macro')
print("Macro F1 Score:", macro_f1)


In [ ]:
y_true=output_ground_truth
y_pred=output_model

precision = precision_score(y_true, y_pred, pos_label=1)
recall = recall_score(y_true, y_pred, pos_label=1)
f1 = f1_score(y_true, y_pred, pos_label=1)

print("Precision for class 1:", precision)
print("Recall for class 1:", recall)
print("F1 Score for class 1:", f1)

cm = confusion_matrix(y_true, y_pred)

TN, FP, FN, TP = cm.ravel()
print("True Positives (TP):", TP)
print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)

In [ ]:
y_true=output_ground_truth
y_pred=output_model

precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')
accuracy = accuracy_score(y_true, y_pred)

print("Precision (Macro):", precision)
print("Recall (Macro):", recall)
print("F1 Score (Macro):", f1)
print("Accuracy:", accuracy)

In [ ]:
sum_correct=0

for key,info in saving_response_valid.items():
    sum_yes=0
    for i in range(len(info)):
        if info[i][2]=='Yes.':
            sum_yes+=1
            
    if info[i][0]['label']==str(4) and sum_yes>=2:
        sum_correct+=1
        
    elif info[i][0]['label']==str(0) and sum_yes<2:
        sum_correct+=1
        
round(sum_correct/len(saving_response_valid)*100)

In [ ]:
baseline_question[0]

In [ ]:
response = ollama.generate(model=model_name, prompt='How many patches in this images? Are they similar?', images=[file_dict[0]], options=options)

In [ ]:
response['response']

In [ ]:
path=file_dict.get(100)
print(path)
display_image(path)


In [ ]:
file_dict[100]
print(test[str(100)]['label'])

In [ ]:
response = ollama.generate(model=model_name, prompt='How are you?', options=options)
print(response['respone'])

In [ ]:
response = ollama.generate(model=model_name, prompt='Calculate the following: 5+6*8+12213', options=options)


In [ ]:
print(response['response'])